# Etapa B: clasificador supervisado con transferencia y atención

En la Etapa A aprendimos qué es un remitente **normal** sin mirar una sola etiqueta.
En este notebook damos el segundo paso: entrenamos un clasificador supervisado que
ahora sí usa las dos clases, reutilizando el encoder que ya entrenamos.

Aquí producimos tres cosas:

1. **Transfer learning:** el encoder no arranca de cero, arranca sabiendo cómo se ve
   un historial normal.
2. **Atención:** el modelo señala *qué transacción* del historial disparó la alerta.
   Es lo que vuelve auditable una decisión de cumplimiento.
3. **Experimento de ablación:** entrenamos el mismo modelo con el encoder preentrenado
   y con el encoder aleatorio, y los comparamos. Sin esa comparación no tendríamos
   forma de afirmar que la Etapa A sirvió de algo.

Las dos semanas que definen esta etapa son la 5, Attention, de donde sale el mecanismo que
nos permite explicar cada alerta, y la 9, Transfer Learning y Fine-Tuning, de donde sale la
estrategia para reutilizar el encoder en lugar de reentrenarlo.

**Consume:** `artifacts/{train,val,test}.npz` y `artifacts/encoder.pt`.
**Produce:** `stage_b_model.pt`, `ablation_results.csv`, `stage_b_threshold.json`
y `fusion_model.pkl`.

In [1]:
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/Vann06/Deep-Learning.git"
BRANCH = "Proyecto2"

if "google.colab" in sys.modules:
    if not Path("src").exists():
        subprocess.check_call(["git", "clone", "--quiet", "--branch", BRANCH, REPO, "repo"])
        get_ipython().run_line_magic("cd", "repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print(f"Repositorio ({BRANCH}) y dependencias listas en Colab")
else:
    print("Entorno local: dependencias existentes")

Entorno local: dependencias existentes


In [2]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
assert (ROOT / "src").exists(), "Ejecutar desde la raíz del repositorio"
ARTIFACTS = ROOT / "artifacts"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

from src.data.dataset import load_split
from src.evaluation.anomaly import anomaly_metrics, reconstruction_scores, select_threshold
from src.evaluation.metrics import classification_metrics, run_ablation, summarize_ablation
from src.models.autoencoder import SequenceAutoencoder
from src.models.classifier import (
    AMLClassifier,
    ClassifierConfig,
    build_classifier,
    load_classifier,
    make_supervised_loader,
    predict_aml,
    predict_scores,
    save_classifier,
)
from src.models.encoder import EncoderConfig, load_encoder
from src.utils import get_device, set_seed

start = time.perf_counter()
set_seed(42)
device = get_device()
print("Raíz:", ROOT)
print("Dispositivo:", device)

Raíz: C:\Users\richi\Documents\2026_S2_Local\Deep-Learning
Dispositivo: cpu


## 1. Ahora sí usamos las dos clases

Antes de cargar los datos vale aclarar algo que nos costó tener claro al inicio: **la
máscara nunca tuvo que ver con las etiquetas**. La máscara marca qué pasos de la secuencia
son transacciones reales y cuáles son relleno hasta llegar a 32. La seguimos usando
exactamente igual aquí, y si la quitáramos, la atención repartiría peso sobre transacciones
que no existen.

Lo que cambia respecto de la Etapa A es otra cosa: en aquel autoencoder elegimos no
entrenar con las secuencias positivas, aunque sí las usamos para fijar el umbral y para
medir. El clasificador de esta etapa entrena con las dos clases.

In [3]:
splits = {name: load_split(ARTIFACTS / f"{name}.npz") for name in ("train", "val", "test")}
feature_names = json.loads((ARTIFACTS / "feature_names.json").read_text(encoding="utf-8"))
num_features = len(feature_names)
train, val, test = splits["train"], splits["val"], splits["test"]

conteos = pd.DataFrame(
    [
        {
            "split": nombre,
            "secuencias": len(s["y"]),
            "normales": int((s["y"] == 0).sum()),
            "sospechosos": int((s["y"] == 1).sum()),
            "tasa_positiva": float(s["y"].mean()),
        }
        for nombre, s in splits.items()
    ]
)
display(conteos)

assert train["y"].sum() > 0, "La Etapa B necesita las secuencias positivas de TRAIN"
print(f"La Etapa A entrenó con {int((train['y'] == 0).sum()):,} normales de TRAIN.")
print(f"La Etapa B entrena con las {len(train['y']):,} secuencias completas de TRAIN.")
print(f"Features por transacción: {num_features}")

,split,secuencias,normales,sospechosos,tasa_positiva
0,train,40551,38620,1931,0.047619
1,val,8673,8260,413,0.047619
2,test,8694,8280,414,0.047619


La Etapa A entrenó con 38,620 normales de TRAIN.
La Etapa B entrena con las 40,551 secuencias completas de TRAIN.
Features por transacción: 49


**Lectura:** entran 1,931 secuencias positivas que la Etapa A nunca vio en su
entrenamiento. La tasa positiva es 4.76% en los tres splits, heredada del submuestreo
20:1 que documentamos en `docs/decisions.md §5`, no de la prevalencia real. Con ese
desbalance, *acertar el 95.2%* se consigue prediciendo "todo normal", así que
descartamos la exactitud como métrica desde el inicio.

## 2. Arquitectura: encoder, atención y cabeza

Con las dos clases ya cargadas, armamos el modelo. Son tres piezas encadenadas: el encoder que viene de la Etapa A, la atención que decide qué transacción pesa más, y una cabeza que convierte ese resumen en una probabilidad.

```
x [B,32,49] ──► SequenceEncoder ──► hidden_states [B,32,64]
                                          │
                              AdditiveAttention(mask)
                                          │
                          context [B,64]  +  weights [B,32]
                                          │
                     Linear(64→32) → ReLU → Dropout → Linear(32→1)
                                          │
                                      logit [B]
```

| Decisión | Por qué la tomamos |
|---|---|
| Atención **aditiva** (Bahdanau) | Aprende su propia proyección antes de puntuar. Con `hidden_size=64` y secuencias cortas nos resulta más estable e interpretable que el producto punto |
| Enmascarar **antes** del softmax | Si enmascaráramos después, el padding recibiría probabilidad y el mapa de calor señalaría transacciones inexistentes |
| La cabeza consume **solo** el contexto, no el latente | Así la probabilidad es íntegramente una suma ponderada de representaciones por transacción y los pesos explican **todo** el output. Concatenar el latente dejaría señal fuera del heatmap y volvería la interpretabilidad una verdad a medias |
| `SequenceEncoder` importado, no redefinido | Los dos brazos de la ablación deben diferir **solo** en la inicialización |

La atención aditiva viene de la Semana 5, y esa idea tiene un recorrido que explica
nuestra elección. El mecanismo nació como una forma de que las redes recurrentes dejaran de
comprimir toda la secuencia en un solo vector, y los Transformers de la Semana 6 son el
paso siguiente,
quitarle la parte recurrente y quedarse solo con la atención. Nosotros usamos la
combinación original, recurrencia más atención, porque con 40,551 secuencias de mediana 15
no tenemos el volumen de datos que un Transformer necesita.

In [4]:
config = ClassifierConfig(encoder=EncoderConfig(num_features=num_features))
set_seed(42)
demo = AMLClassifier(config)

indices = np.arange(4)
x_demo = torch.from_numpy(train["X"][indices])
lengths_demo = torch.from_numpy(train["lengths"][indices].astype(np.int64))
mask_demo = torch.from_numpy(train["mask"][indices])

with torch.no_grad():
    logits_demo, weights_demo = demo(x_demo, lengths_demo, mask_demo)

print("logits:", tuple(logits_demo.shape))
print("pesos de atención:", tuple(weights_demo.shape))
print("¿los pesos suman 1 por secuencia?", bool(torch.allclose(weights_demo.sum(1), torch.ones(4), atol=1e-6)))
print("peso máximo asignado a padding:", float(weights_demo[~mask_demo].abs().max()))
print("longitudes reales del lote:", lengths_demo.tolist())

total_params = sum(p.numel() for p in demo.parameters())
encoder_params = sum(p.numel() for p in demo.encoder.parameters())
print(f"\nParámetros totales: {total_params:,} (encoder heredable: {encoder_params:,})")

logits: (4,)
pesos de atención: (4, 32)
¿los pesos suman 1 por secuencia? True
peso máximo asignado a padding: 0.0
longitudes reales del lote: [7, 15, 5, 6]

Parámetros totales: 28,385 (encoder heredable: 24,160)


**Verificación:** los pesos suman 1 sobre los pasos reales y valen **exactamente 0** sobre
el padding. Con eso confirmado, el heatmap de la interfaz tendrá una casilla por cada
transacción que de verdad ocurrió.

## 3. Transfer learning: de dónde arranca el encoder

Ya que el modelo arma bien sus piezas y enmascara como esperábamos, toca conectarlo con lo
que dejó la Etapa A.

El brazo preentrenado carga los pesos de `encoder.pt`, la GRU que entrenamos sobre 38,620
remitentes normales. El brazo de control instancia la misma clase con la misma
configuración, pero con la inicialización aleatoria de PyTorch.

**Learning rate diferenciado:** ajustamos el encoder con `lr=1e-4` y la atención y la
cabeza con `lr=1e-3`. Queremos adaptar la representación a la tarea supervisada sin borrar
de un golpe la normalidad aprendida: con un lr alto sobre todo el modelo, las primeras
épocas destruirían lo que estamos intentando transferir.

Esta es la aplicación directa de la Semana 9, Transfer Learning y Fine-Tuning, donde
teníamos dos estrategias posibles. Congelar el encoder lo dejaría como extractor
de características fijo, entrenando solo atención y cabeza: más rápido, pero la
representación se quedaría optimizada para reconstruir y no para discriminar. Optamos por
fine-tuning con learning rate diferenciado, que adapta el encoder a la tarea supervisada
pero a un ritmo diez veces menor que el resto, para no perder en las primeras épocas lo que
costó 28 minutos aprender.

In [5]:
encoder_ref, encoder_config = load_encoder(ARTIFACTS / "encoder.pt", map_location=device)
print("EncoderConfig guardada en la Etapa A:", encoder_config)
print("¿coincide con la del clasificador?", encoder_config == config.encoder)

brazo_pretrained = build_classifier(config, pretrained_encoder_path=ARTIFACTS / "encoder.pt")
brazo_scratch = build_classifier(config, pretrained_encoder_path=None)

referencia = encoder_ref.state_dict()
hereda_pretrained = all(
    torch.equal(referencia[k], brazo_pretrained.encoder.state_dict()[k]) for k in referencia
)
hereda_scratch = all(
    torch.equal(referencia[k], brazo_scratch.encoder.state_dict()[k]) for k in referencia
)
print(f"\nBrazo 'pretrained' hereda los {len(referencia)} tensores de la Etapa A: {hereda_pretrained}")
print(f"Brazo 'scratch' los hereda: {hereda_scratch}   (debe ser False)")

EncoderConfig guardada en la Etapa A: EncoderConfig(num_features=49, hidden_size=64, latent_size=32, num_layers=1, dropout=0.0)
¿coincide con la del clasificador? True

Brazo 'pretrained' hereda los 6 tensores de la Etapa A: True
Brazo 'scratch' los hereda: False   (debe ser False)


**Lectura:** comprobamos antes de entrenar que la transferencia ocurre de verdad y que
el brazo de control no la recibe. Si esto fallara, nuestra ablación mediría ruido.

## 4. Función de pérdida y criterio de parada

Con la transferencia confirmada, nos falta decidir cómo entrenar.

| Hiperparámetro | Valor | Justificación |
|---|---|---|
| Pérdida | `BCEWithLogitsLoss(pos_weight=20)` | Lo tomamos de `metadata.json → imbalance.train_pos_weight_recommended`. Compensa el submuestreo 20:1: sin él, el gradiente casi no empuja hacia la clase positiva |
| Optimizador | Adam, lr 1e-4 en el encoder y 1e-3 en atención y cabeza | Fine-tune conservador del encoder transferido |
| Batch | 256 | Igual que en la Etapa A, estabiliza el gradiente sin saturar memoria |
| Épocas máximas | 40 | Techo con early stopping, no un objetivo |
| *Early stopping* | Paciencia 8 sobre el **PR-AUC de VALIDATION** | Ver abajo |
| Semillas | 42, 7, 1234 | Tres corridas por brazo para separar señal de ruido |

**Por qué nuestro early stopping mira PR-AUC y no la pérdida.** Es la lección que nos dejó
la Etapa A: entrenar hasta minimizar el error de reconstrucción mejoró la pérdida pero
empeoró la separación entre clases. Menor pérdida no implica mejor detección, así que aquí
seleccionamos el checkpoint por la métrica que nos importa con 4.76% de positivos.

Casi todo lo de esta tabla sale de la Semana 8: la entropía cruzada binaria como pérdida
para clasificación, `pos_weight` como forma de tratar el desbalance desde la función de
pérdida, el dropout de la cabeza como regularización, Adam como optimizador y el early
stopping como criterio de parada.

Sobre el desbalance había otro camino posible dentro del curso: usar una red generativa de
la Semana 7 para crear secuencias positivas sintéticas y balancear el dataset. Lo
descartamos porque una GAN aprendería a imitar las 1,931 secuencias de lavado que ya
tenemos, y el riesgo en detección es justamente lo contrario, las tipologías que nadie ha
visto. Ponderar la pérdida no inventa casos que no existen.

In [6]:
BRAZOS = {"pretrained": ARTIFACTS / "encoder.pt", "scratch": None}
SEMILLAS = (42, 7, 1234)

ablacion_start = time.perf_counter()
results, runs = run_ablation(
    splits,
    config,
    arms=BRAZOS,
    seeds=SEMILLAS,
    device=device,
    batch_size=256,
    epochs=40,
    patience=8,
    encoder_lr=1e-4,
    head_lr=1e-3,
    pos_weight=20.0,
)
ablacion_minutos = (time.perf_counter() - ablacion_start) / 60
print(f"{len(BRAZOS) * len(SEMILLAS)} corridas completadas en {ablacion_minutos:.2f} minutos")

6 corridas completadas en 39.54 minutos


In [7]:
historia = runs[("pretrained", 42)]["history"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(historia["train_loss"], color="tab:blue")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("BCE ponderada (train)")
axes[0].set_title("Pérdida de entrenamiento")

axes[1].plot(historia["val_pr_auc"], color="tab:green")
axes[1].axvline(historia["best_epoch"], color="grey", linestyle="--", label="Checkpoint elegido")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("PR-AUC (validation)")
axes[1].set_title("Criterio de early stopping")
axes[1].legend()

fig.suptitle("Brazo principal: encoder preentrenado, semilla 42")
fig.tight_layout()
fig.savefig(FIGURES / "stage_b_training_curve.png", dpi=110)
plt.show()

print(f"Épocas ejecutadas: {historia['epochs_run']} (mejor época: {historia['best_epoch']})")
print(f"PR-AUC de validación en la mejor época: {historia['best_val_pr_auc']:.4f}")

Épocas ejecutadas: 40 (mejor época: 38)
PR-AUC de validación en la mejor época: 0.7647


C:\Users\richi\AppData\Local\Temp\ipykernel_27488\3269388214.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura de las curvas:** el PR-AUC de validación llega a **0.7647** en la época 38 de 40,
y seguía subiendo cuando se acabó el presupuesto: nuestro *early stopping* con paciencia 8
nunca llegó a activarse. Las 6 corridas terminaron igual, con la mejor época entre la 36 y
la 39 de 40.

Es el mismo patrón que vimos en la Etapa A: el modelo se detuvo por tope de épocas, no por
convergencia. Con más presupuesto probablemente rendiría algo mejor. Fijamos el techo de 40
para que el notebook completo corra en menos de 30 minutos, no porque el modelo hubiera
dejado de aprender.

## 5. Experimento de demostración: ¿aporta valor la arquitectura de dos etapas?

Aquí está la pregunta que decide si nuestro diseño se justifica. Comparamos dos brazos:

- **Preentrenado (dos etapas):** el encoder arranca con los pesos que la Etapa A aprendió
  sobre comportamiento normal.
- **Desde cero (línea base):** el mismo clasificador supervisado, con el encoder
  inicializado al azar. Es decir, el sistema **sin Etapa A**.

Todo lo demás lo dejamos idéntico entre brazos: misma arquitectura, mismos datos, mismo
orden de batches, mismo optimizador, mismos hiperparámetros y la misma semilla por par. La
única variable que manipulamos es de dónde parten los pesos del encoder, y eso es lo que
nos permite atribuirle la diferencia.

Entrenamos cada brazo con **tres semillas** (42, 7 y 1234) y reportamos media y desviación
estándar. Con una sola corrida por brazo, una diferencia pequeña sería indistinguible del
azar de la inicialización.

### Métrica elegida y por qué

**Métrica principal: PR-AUC** (*average precision*). **Secundaria: ROC-AUC.**

Con 4.76% de positivos, la elección cambia bastante lo que se ve:

| Métrica | Por qué la usamos o la descartamos |
|---|---|
| **Exactitud** | **Descartada.** Predecir "todo normal" ya acierta el 95.24%. Un modelo inútil se vería excelente |
| **F1** | **Descartada para comparar modelos.** Depende de un umbral, así que compararía decisiones de corte en vez de la calidad de los modelos. Sí la usamos más adelante, para elegir el umbral operativo |
| **ROC-AUC** | **Secundaria.** Es optimista con clases desbalanceadas: la tasa de falsos positivos se calcula sobre 8,280 negativos, así que cientos de falsas alarmas apenas la mueven. La reportamos por comparabilidad con la literatura |
| **PR-AUC** | **Principal.** Precisión y exhaustividad se calculan ambas sobre la clase minoritaria, que es la que nos interesa. Penaliza los falsos positivos en proporción a las alertas reales y no al total de clientes, que es el costo que siente un equipo de cumplimiento |

La referencia de azar para PR-AUC es la prevalencia misma, **0.0476**, y contra eso leemos
cuánto aporta cada brazo.

In [8]:
display(results.round(4))

resumen = summarize_ablation(results)
display(resumen.round(4))

media_pre = resumen.loc["pretrained", ("test_pr_auc", "mean")]
media_scr = resumen.loc["scratch", ("test_pr_auc", "mean")]
desv_pre = resumen.loc["pretrained", ("test_pr_auc", "std")]
desv_scr = resumen.loc["scratch", ("test_pr_auc", "std")]
delta = media_pre - media_scr
traslape = abs(delta) < (desv_pre + desv_scr)

print(f"PR-AUC en TEST, preentrenado: {media_pre:.4f} ± {desv_pre:.4f}")
print(f"PR-AUC en TEST, desde cero:   {media_scr:.4f} ± {desv_scr:.4f}")
print(f"Diferencia (preentrenado - desde cero): {delta:+.4f}")
print(f"¿Las desviaciones se traslapan? {traslape}")

,arm,seed,epochs_run,best_epoch,val_roc_auc,val_pr_auc,test_roc_auc,test_pr_auc,train_minutes
0,pretrained,42,40,38,0.9575,0.7647,0.9716,0.8070,8.6686
1,pretrained,7,40,38,0.9579,0.7543,0.9741,0.8156,8.4540
2,pretrained,1234,40,38,0.9569,0.7607,0.9724,0.8096,7.8925
3,scratch,42,40,39,0.9546,0.7096,0.9616,0.7638,5.3782
4,scratch,7,40,36,0.9564,0.7406,0.9627,0.7811,4.5304
5,scratch,1234,40,38,0.9513,0.7289,0.9610,0.7804,4.5056


val_pr_auc         test_pr_auc         test_roc_auc        
                 mean     std        mean     std         mean     std
arm                                                                   
pretrained     0.7599  0.0053      0.8107  0.0044       0.9727  0.0013
scratch        0.7264  0.0157      0.7751  0.0098       0.9618  0.0008

PR-AUC en TEST — preentrenado: 0.8107 ± 0.0044
PR-AUC en TEST — desde cero:   0.7751 ± 0.0098
Diferencia (preentrenado - desde cero): +0.0356
¿Las desviaciones se traslapan? False


**Lectura de la ablación: el preentrenamiento sí aportó, y lo podemos sostener.**

| Brazo | PR-AUC en VAL | PR-AUC en TEST | ROC-AUC en TEST |
|---|---|---|---|
| Encoder preentrenado (Etapa A) | 0.7599 ± 0.0053 | **0.8107 ± 0.0044** | 0.9727 ± 0.0013 |
| Encoder desde cero | 0.7264 ± 0.0157 | 0.7751 ± 0.0098 | 0.9618 ± 0.0008 |

La diferencia en TEST es de **+0.0356 de PR-AUC**, y las desviaciones entre semillas
**no se traslapan**, así que la ventaja es más grande que el ruido de inicialización.
El brazo preentrenado ganó en las tres semillas, no en una corrida afortunada.

Un detalle secundario apunta en la misma dirección: el brazo preentrenado es además más
*estable*, con desviación 0.0044 contra 0.0098, consistente con arrancar desde una
representación ya formada en vez de desde ruido.

La magnitud es moderada, y era lo que esperábamos: la Etapa A optimizó reconstrucción y
no discriminación, y su propio PR-AUC fue apenas 0.308. Lo que transfiere es una
inicialización informada del encoder, no un detector ya entrenado.

## 6. Umbral del clasificador

Con la ablación resuelta, nos falta decidir a partir de qué probabilidad levantamos una
alerta.

Conservamos el modelo del brazo preentrenado con **la semilla de mejor PR-AUC en
VALIDATION**. Elegirla por TEST sería leakage, porque el conjunto de prueba lo tocamos una
sola vez, más abajo.

Elegimos el umbral con las mismas funciones que usamos en la Etapa A (`select_threshold`),
para tener una sola metodología en todo el proyecto y no dos criterios paralelos.

In [9]:
filas_pretrained = results[results["arm"] == "pretrained"]
mejor = filas_pretrained.loc[filas_pretrained["val_pr_auc"].idxmax()]
mejor_clave = (mejor["arm"], int(mejor["seed"]))
print(f"Modelo conservado: brazo '{mejor_clave[0]}', semilla {mejor_clave[1]} "
      f"(PR-AUC de VAL = {mejor['val_pr_auc']:.4f})")

modelo_final = build_classifier(config)
modelo_final.load_state_dict(runs[mejor_clave]["state_dict"])
modelo_final.to(device).eval()

val_loader = make_supervised_loader(val, batch_size=512)
test_loader = make_supervised_loader(test, batch_size=512)
prob_val, logit_val, y_val = predict_scores(modelo_final, val_loader, device)

candidatos = [
    select_threshold(prob_val, y_val, method="f1_max"),
    select_threshold(prob_val, y_val, method="budget", budget=0.05),
]
display(pd.DataFrame(candidatos)[["method", "threshold", "precision", "recall", "f1", "alert_rate"]])

Modelo conservado: brazo 'pretrained', semilla 42 (PR-AUC de VAL = 0.7647)


,method,threshold,precision,recall,f1,alert_rate
0,f1_max,0.952144,0.940000,0.569007,0.708899,0.028825
1,budget,0.791152,0.654378,0.687651,0.670602,0.050040


**Justificación del umbral:** tomamos el de **F1 máximo**, por la misma razón que en la
Etapa A: con 4.76% de positivos, la exactitud no informa y ROC-AUC resulta optimista
frente al desbalance. El candidato de presupuesto fijo, revisar el 5% de mayor score, lo
reportamos como su traducción operativa directa, porque es el que un equipo de
cumplimiento con capacidad limitada usaría en la práctica.

## 7. Cómo combinamos las señales de las dos etapas

Tenemos dos señales funcionando por separado. Falta ver si juntas rinden más.

Cada una mide algo distinto. El score de la Etapa A dice *"este historial no se parece a lo
normal"*; el logit de la Etapa B dice *"este historial se parece a los casos de lavado
etiquetados"*. Un remitente puede ser raro sin ser sospechoso, y al revés.

Nuestro mecanismo de combinación es una **fusión tardía**: una regresión logística sobre
`[score_A, logit_B]` estandarizados, ajustada en VALIDATION. Cada etapa se entrena por
separado y solo al final aprendemos cómo pesarlas. Así mantenemos ambos modelos
independientes y auditables, a diferencia de un modelo conjunto donde no podríamos atribuir
la decisión a ninguna de las dos.

En la celda siguiente medimos si esa combinación aporta, en vez de asumirlo.

In [10]:
checkpoint_a = torch.load(ARTIFACTS / "stage_a_model.pt", map_location=device, weights_only=False)
stage_a = SequenceAutoencoder(EncoderConfig(**checkpoint_a["config"]))
stage_a.load_state_dict(checkpoint_a["state_dict"])

score_a_val = reconstruction_scores(stage_a, val["X"], val["mask"], val["lengths"], device=device)
score_a_test = reconstruction_scores(stage_a, test["X"], test["mask"], test["lengths"], device=device)

fusion = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(class_weight="balanced", max_iter=1000)),
    ]
)
X_fusion_val = np.column_stack([score_a_val, logit_val])
fusion.fit(X_fusion_val, y_val)
prob_fusion_val = fusion.predict_proba(X_fusion_val)[:, 1]

coeficientes = fusion.named_steps["lr"].coef_[0]
print(f"Peso aprendido sobre score_A: {coeficientes[0]:+.4f}")
print(f"Peso aprendido sobre logit_B: {coeficientes[1]:+.4f}")

comparacion_val = pd.DataFrame(
    [
        {"señal": "Etapa A (autoencoder)", **anomaly_metrics(score_a_val, y_val)},
        {"señal": "Etapa B (clasificador)", **anomaly_metrics(prob_val, y_val)},
        {"señal": "Fusión tardía", **anomaly_metrics(prob_fusion_val, y_val)},
    ]
)
display(comparacion_val.round(4))

Peso aprendido sobre score_A: +0.2721
Peso aprendido sobre logit_B: +2.8085


,señal,roc_auc,pr_auc,n_sequences,n_positive
0,Etapa A (autoencoder),0.7507,0.2719,8673,413
1,Etapa B (clasificador),0.9575,0.7647,8673,413
2,Fusión tardía,0.9583,0.7650,8673,413


**Lectura:** la fusión casi no mueve la aguja. En VALIDATION obtiene 0.7650 de PR-AUC
contra 0.7647 de la Etapa B sola, y ese margen mínimo es además optimista porque ajustamos
la fusión sobre ese mismo conjunto.

Los pesos que aprendió lo explican: **+0.272 sobre `score_A`** contra **+2.809 sobre
`logit_B`**, o sea que la regresión se apoya diez veces más en el clasificador. Tiene
sentido, porque la Etapa B ya *contiene* la información de la Etapa A: su encoder es el de
la Etapa A, afinado. El score de reconstrucción no agrega una vista independiente, solo una
versión más pobre de la misma.

El veredicto definitivo está en TEST, abajo.

## 8. Evaluación final en TEST

Ya con el umbral fijado y la fusión ajustada, llega el momento de la prueba final.

Tocamos TEST **una sola vez**, con el modelo, el umbral y la fusión ya congelados. No
reajustamos nada después de ver estos números.

Declaración de honestidad metodológica: usamos VALIDATION para tres cosas, el early
stopping, la selección del umbral y el ajuste de la fusión. Es práctica estándar, pero
implica que nuestras métricas de VAL están algo optimistas. Las de TEST no.

In [11]:
prob_test, logit_test, y_test = predict_scores(modelo_final, test_loader, device)
X_fusion_test = np.column_stack([score_a_test, logit_test])
prob_fusion_test = fusion.predict_proba(X_fusion_test)[:, 1]

comparacion_test = pd.DataFrame(
    [
        {"señal": "Etapa A (autoencoder)", **anomaly_metrics(score_a_test, y_test)},
        {"señal": "Etapa B (clasificador)", **anomaly_metrics(prob_test, y_test)},
        {"señal": "Fusión tardía", **anomaly_metrics(prob_fusion_test, y_test)},
    ]
)
display(comparacion_test.round(4))

umbral_elegido = candidatos[0]
metricas_test = classification_metrics(prob_test, y_test, umbral_elegido["threshold"])
print(f"\nUmbral congelado desde VAL (método={umbral_elegido['method']}): {umbral_elegido['threshold']:.5f}")
print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in metricas_test.items()}, indent=2))

matriz = pd.DataFrame(
    [
        [metricas_test["true_negative"], metricas_test["false_positive"]],
        [metricas_test["false_negative"], metricas_test["true_positive"]],
    ],
    index=["real_normal", "real_sospechoso"],
    columns=["pred_normal", "pred_sospechoso"],
)
display(matriz)

,señal,roc_auc,pr_auc,n_sequences,n_positive
0,Etapa A (autoencoder),0.7671,0.3084,8694,414
1,Etapa B (clasificador),0.9716,0.8070,8694,414
2,Fusión tardía,0.9714,0.8035,8694,414



Umbral congelado desde VAL (método=f1_max): 0.95214
{
  "threshold": 0.9521,
  "precision": 0.9401,
  "recall": 0.6063,
  "f1": 0.7372,
  "alert_rate": 0.0307,
  "true_positive": 251,
  "false_positive": 16,
  "false_negative": 163,
  "true_negative": 8264
}


,pred_normal,pred_sospechoso
real_normal,8264,16
real_sospechoso,163,251


**Lectura de TEST.**

| Señal | ROC-AUC | PR-AUC |
|---|---|---|
| Etapa A (autoencoder solo) | 0.7671 | 0.3084 |
| **Etapa B (clasificador)** | **0.9716** | **0.8070** |
| Fusión tardía | 0.9714 | 0.8035 |

La supervisión con atención multiplica por **2.6** el PR-AUC de la detección de anomalías
pura, 0.807 contra 0.308. Es el salto más grande del proyecto.

La fusión, en cambio, queda por debajo de la Etapa B sola: 0.8035 contra 0.8070. Confirma
lo que insinuaba VALIDATION, que la pequeña mejora de allá era sobreajuste al conjunto
donde la ajustamos.

### Nuestra predicción final

Con esa evidencia definimos así el sistema que entregamos:

| Salida | De dónde viene |
|---|---|
| Decisión de alerta | probabilidad de la **Etapa B** |
| Contexto de anomalía | z-score de la **Etapa A**, del tipo "2.7 sigmas sobre el cliente normal" |
| Explicación por transacción | pesos de atención de la **Etapa B** |

Usamos las dos etapas, pero no las fusionamos en un solo número, porque medimos esa fusión
y no mejoraba. La Etapa B decide y la Etapa A aporta el contexto que el analista ve junto a
la alerta.

En carga de trabajo, con el umbral congelado desde VAL (0.95214): de cada 1,000 remitentes
alertamos **31**, y **94% de esas alertas son casos reales**, 251 verdaderos positivos
contra 16 falsos. El costo está del otro lado, porque se nos escapan 163 de los 414
sospechosos, un recall de 60.6%. Pocas alertas y casi todas buenas suele ser preferible a
inundar al equipo de revisión, pero es una decisión de negocio que el umbral deja explícita
y ajustable.

## 9. ¿A qué transacciones les pone atención el modelo?

Ya sabemos que el clasificador acierta. Lo que todavía no sabemos es si su explicación
sirve.

Nuestros `.npz` guardan `transaction_y`, la etiqueta por transacción y no solo por
remitente, así que podemos medirlo en lugar de suponerlo: **¿la atención se concentra en
las transacciones que de verdad están marcadas como lavado?**

Usamos un *lift*: la masa de atención que cae sobre las transacciones etiquetadas, dividida
entre la masa que le tocaría si la atención fuera uniforme.

- lift = 1 significa que la atención no distingue nada.
- lift > 1 significa que la atención señala las transacciones correctas.

In [12]:
def pesos_de_atencion(modelo, arrays, indices, device, batch_size=512):
    # Pesos de atención para un subconjunto de secuencias.
    modelo.eval()
    salida = np.zeros((len(indices), arrays["X"].shape[1]), dtype=np.float32)
    with torch.no_grad():
        for inicio in range(0, len(indices), batch_size):
            trozo = indices[inicio : inicio + batch_size]
            x = torch.from_numpy(arrays["X"][trozo]).to(device)
            m = torch.from_numpy(arrays["mask"][trozo]).to(device)
            l = torch.from_numpy(arrays["lengths"][trozo].astype(np.int64))
            _, w = modelo(x, l, m)
            salida[inicio : inicio + len(trozo)] = w.cpu().numpy()
    return salida


positivos = np.flatnonzero(test["y"] == 1)
atencion_pos = pesos_de_atencion(modelo_final, test, positivos, device)

lifts = []
for fila, idx in enumerate(positivos):
    largo = int(test["lengths"][idx])
    pesos = atencion_pos[fila, :largo]
    marcadas = test["transaction_y"][idx, :largo] == 1
    if marcadas.any() and not marcadas.all():
        esperado = marcadas.sum() / largo
        lifts.append(float(pesos[marcadas].sum() / esperado))

lifts = np.array(lifts)
print(f"Secuencias positivas evaluadas: {len(lifts):,}")
print(f"Lift medio de atención sobre transacciones etiquetadas: {lifts.mean():.3f}")
print(f"Mediana: {np.median(lifts):.3f}   |   % con lift > 1: {100 * (lifts > 1).mean():.1f}%")

Secuencias positivas evaluadas: 414
Lift medio de atención sobre transacciones etiquetadas: 5.184
Mediana: 0.772   |   % con lift > 1: 45.4%


In [13]:
orden = positivos[np.argsort(-prob_test[positivos])][:2]
fig, axes = plt.subplots(len(orden), 1, figsize=(11, 2.4 * len(orden)))

for eje, idx in zip(np.atleast_1d(axes), orden):
    largo = int(test["lengths"][idx])
    fila = int(np.flatnonzero(positivos == idx)[0])
    pesos = atencion_pos[fila, :largo]
    marcadas = test["transaction_y"][idx, :largo] == 1
    eje.bar(np.arange(largo), pesos,
                     color=["tab:red" if m else "tab:blue" for m in marcadas])
    eje.set_title(
        f"Remitente {test['sender_id'][idx]}, p={prob_test[idx]:.3f}, "
        f"rojo = transacción etiquetada como lavado",
        fontsize=9,
    )
    eje.set_xlabel("Transacción en el historial")
    eje.set_ylabel("Atención")

fig.tight_layout()
fig.savefig(FIGURES / "stage_b_attention_heatmap.png", dpi=110)
plt.show()

C:\Users\richi\AppData\Local\Temp\ipykernel_27488\1190829951.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Lectura: la atención acierta en algunos casos y en otros no.**

| Métrica sobre las 414 secuencias positivas de TEST | Valor |
|---|---|
| Lift **medio** | 5.184 |
| Lift **mediano** | **0.772** |
| Fracción con lift > 1 | 45.4% |

El promedio de 5.18 parece muy bueno, pero la distribución tiene cola pesada y unas pocas
secuencias con lift altísimo lo inflan. La **mediana de 0.772 está por debajo de 1**, y
solo el 45.4% de los casos supera lo que daría una atención uniforme.

Así que el resultado es mixto: en una minoría de secuencias la atención se concentra con
fuerza en las transacciones realmente etiquetadas, pero en el caso típico no las prioriza
por encima del azar. El modelo clasifica muy bien, con PR-AUC 0.807, usando señal
distribuida en todo el historial y no necesariamente la transacción concreta que el dataset
marcó.

Esto acota qué podemos afirmar con el heatmap: sirve para mostrar en qué se fijó el modelo,
pero no prueba que señale la transacción culpable. Hay una causa plausible que nos gustaría
investigar en el análisis de casos: la etiqueta por transacción marca el movimiento
*detectado*, mientras que el patrón de lavado suele estar en las transacciones que lo
rodean, con estructuración, fragmentación y destinos nuevos.

Para ese análisis nos conviene elegir ejemplos de ambos lados de esta distribución y
discutir la diferencia, en vez de mostrar solo los que salen bien.

## 10. Artefactos y contrato para el MVP

Para cerrar, guardamos lo que la interfaz va a consumir y comprobamos que recargarlo desde disco devuelva la misma probabilidad.

```python
from src.evaluation.anomaly import get_anomaly_score   # Etapa A
from src.models.classifier import predict_aml          # Etapa B

probabilidad, pesos = predict_aml(secuencia, longitud_real)
```

`predict_aml` devuelve la tupla exacta que `docs/DIVISION.md` le promete a la app, con
los pesos ya recortados a la longitud real del historial, una casilla por transacción
que existió.

El MVP usa las dos funciones: `predict_aml` da la decisión y el heatmap, y
`get_anomaly_score` aporta el contexto en sigmas. No carga `fusion_model.pkl`, que
guardamos solo como registro del experimento de combinación.

In [14]:
save_classifier(ARTIFACTS / "stage_b_model.pt", modelo_final, config)
results.to_csv(ARTIFACTS / "ablation_results.csv", index=False)

reporte_umbral = {
    "config": {
        "encoder": {
            "num_features": config.encoder.num_features,
            "hidden_size": config.encoder.hidden_size,
            "latent_size": config.encoder.latent_size,
            "num_layers": config.encoder.num_layers,
            "dropout": config.encoder.dropout,
        },
        "attention_size": config.attention_size,
        "head_hidden": config.head_hidden,
        "dropout": config.dropout,
    },
    "modelo_conservado": {"arm": mejor_clave[0], "seed": mejor_clave[1]},
    "candidatos": {c["method"]: c for c in candidatos},
    "selected": {"method": umbral_elegido["method"], "value": float(umbral_elegido["threshold"])},
    "val_metrics": anomaly_metrics(prob_val, y_val),
    "test_metrics": anomaly_metrics(prob_test, y_test),
    "test_at_threshold": metricas_test,
    "fusion": {
        "features": ["score_A", "logit_B"],
        "ajustada_en": "validation",
        "coef": [float(c) for c in coeficientes],
        "intercept": float(fusion.named_steps["lr"].intercept_[0]),
        "test_metrics": anomaly_metrics(prob_fusion_test, y_test),
    },
    "ablation": json.loads(summarize_ablation(results).round(6).to_json()),
    "attention_lift": {
        "media": float(lifts.mean()),
        "mediana": float(np.median(lifts)),
        "fraccion_mayor_a_1": float((lifts > 1).mean()),
        "n_secuencias": int(len(lifts)),
    },
}
(ARTIFACTS / "stage_b_threshold.json").write_text(
    json.dumps(reporte_umbral, indent=2, ensure_ascii=False), encoding="utf-8"
)
with (ARTIFACTS / "fusion_model.pkl").open("wb") as stream:
    pickle.dump(fusion, stream)

print("Artefactos guardados en", ARTIFACTS)

# Round-trip: recargar desde disco y confirmar que la probabilidad no cambia
from src.models import classifier as classifier_module

classifier_module._STAGE_B_CACHE.clear()
muestra = int(positivos[0])
probabilidad, pesos = predict_aml(
    test["X"][muestra],
    int(test["lengths"][muestra]),
    model_path=ARTIFACTS / "stage_b_model.pt",
    device=device,
)
print(f"\npredict_aml sobre un remitente sospechoso de TEST:")
print(f"  probabilidad: {probabilidad:.5f}")
print(f"  pesos de atención: {pesos.shape[0]} valores (longitud real), suman {pesos.sum():.5f}")
print(f"  probabilidad calculada en memoria: {prob_test[muestra]:.5f}")

round_trip_ok = abs(probabilidad - float(prob_test[muestra])) < 1e-4
print(f"  ¿coincide el round-trip desde disco? {round_trip_ok}")
assert round_trip_ok, "La probabilidad recargada no coincide, revisar guardado/carga"

print(f"\nTiempo total del notebook: {(time.perf_counter() - start) / 60:.2f} minutos")

Artefactos guardados en C:\Users\richi\Documents\2026_S2_Local\Deep-Learning\artifacts

predict_aml sobre un remitente sospechoso de TEST:
  probabilidad: 1.00000
  pesos de atención: 26 valores (longitud real), suman 1.00000
  probabilidad calculada en memoria: 1.00000
  ¿coincide el round-trip desde disco? True

Tiempo total del notebook: 39.62 minutos


## Cierre

**Resultados de la etapa:**

- **Clasificador:** ROC-AUC 0.9716 y PR-AUC 0.8070 en TEST, contra 0.7671 y 0.3084 de la
  Etapa A sola. Con el umbral congelado desde VAL logramos 94% de precisión, 60.6% de
  recall y 31 alertas por cada 1,000 remitentes.
- **Ablación de 6 corridas:** el encoder preentrenado aporta **+0.0356 de PR-AUC** en TEST
  (0.8107 ± 0.0044 contra 0.7751 ± 0.0098), con desviaciones que no se traslapan. Con esto
  queda justificada empíricamente la transferencia desde la Etapa A.
- **Combinación de señales:** implementamos la fusión tardía y no mejora (0.8035 contra
  0.8070). Nuestra predicción final usa la Etapa B para decidir y la Etapa A como contexto
  de anomalía.
- **Atención:** clasifica bien, pero su alineación con las transacciones etiquetadas es
  parcial, con lift mediano 0.772 y 45.4% por encima de uniforme.

**Limitaciones que llevamos al reporte:**

- Las 6 corridas terminaron por tope de 40 épocas y no por convergencia: el early stopping
  nunca se activó y el PR-AUC seguía subiendo.
- Calibramos el umbral sobre un dataset submuestreado 20:1. Con la prevalencia operativa
  real de 0.102% por transacción la precisión caería mucho, así que habría que recalibrarlo
  antes de llevarlo a producción.
- La atención explica dónde miró el modelo, no prueba causalidad sobre la transacción
  marcada.
- Una ventana por remitente, heredada de `01_data_engineering.ipynb`, pierde historia en
  cuentas muy activas.
- Las etiquetas marcan transacciones conocidas, no toda conducta ilícita posible: un falso
  positivo puede ser un caso real sin etiquetar.
- IBM AML es sintético, así que los patrones no representan directamente el corredor de
  remesas Guatemala a Estados Unidos que motiva el proyecto.

**Técnicas del curso aplicadas en el proyecto:**

| Semana | Tema | Cómo la usamos |
|---|---|---|
| 1 | Introducción a Deep Learning | La cabeza clasificadora es un MLP con ReLU y dropout |
| 2 | Redes Neuronales Convolucionales | Descartada: su invarianza a la traslación borra el orden temporal, que aquí importa |
| 3 | Redes Neuronales Recurrentes y LSTM | GRU en el encoder, elegida sobre LSTM por tener menos parámetros |
| 4 | Encoder Decoder Autoencoders | Toda la Etapa A: cuello de botella de 1,568 a 32 y reconstrucción como score |
| 5 | Attention | Atención aditiva enmascarada, que produce el heatmap por transacción |
| 6 | Transformers | Descartada: necesitan más dato y secuencias más largas que las nuestras |
| 7 | Redes Generativas Adversarias | Descartada para el desbalance: preferimos `pos_weight` antes que generar positivos sintéticos |
| 8 | Pérdida, Regularización y Optimización | MSE enmascarada, `BCEWithLogitsLoss(pos_weight=20)`, dropout, Adam, recorte de gradiente y early stopping |
| 9 | Transfer Learning y Fine-Tuning | El diseño de dos etapas, el fine-tune con lr diferenciado y el experimento de ablación que lo valida |

**Qué sigue:** el análisis de interpretabilidad con cinco casos y el reporte escrito, y la
interfaz en Streamlit que consume `get_anomaly_score` y `predict_aml`.